# Assignment 3: request timing

This notebook drives your `submission/request_policy.py` on a scripted road and
shows when it asks the model, how long each answer's command holds, and what it
does through an outage and through quota refusals. It applies the waste,
reaction, and fault-conduct rules of the brief's
[How you are marked](README.md#how-you-are-marked) to that drive.

It runs no simulator and asks no model, so it costs nothing and takes seconds.
The road is a script: it gives the scene DriveBench measures on every control
tick. Recorded answers stand in for the model, and each reaches the car
1.4 s after its request, about as late as Gemini's did in the course's drives.
Everything between your policy and the model is DriveBench's own: the runtime
that calls `should_request`, the request floor below it (one request in flight,
the minimum interval, the budget), the safety floor that accepts each answer,
and the fault wrapper that refuses requests during a fault. The car does not
drive: it moves as the script says, whatever the answers say. So the drive
shows timing, never driving, and it never replaces a drive on Vertex.

Four tasks follow the `TODO` lines in `submission/request_policy.py`. Each ends
with a check, printed as `PASS` or `NOT YET`, that also runs the structural
gates' own checks of your policy. The notebook is never submitted and never
graded, and its figures come from a scripted road, so never cite them as
measurements of your policy on Gemini.

Start it from the repository root:

```bash
uv run --extra notebooks jupyter lab course/assignment-3/request-timing.ipynb
```

After every change to `submission/request_policy.py`, rerun the notebook from
the top.

In [ ]:
import json
import math
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt

from metadrive_starter.config import load_config
from metadrive_starter.faults import FaultKind
from metadrive_starter.gates import CONTROL_LOOP_UNBLOCKED, MONOTONE_RESTRICTION
from metadrive_starter.perception.scene import LaneRelation, LocalScene, TrackedObject
from metadrive_starter.request_timing import FaultWindow, drive_request_policy
from metadrive_starter.submission import PROJECT_ROOT, apply_submission_overlay, load_request_policy
from metadrive_starter.vla import RequestPolicySettings, withhold_scene

SUBMISSION = PROJECT_ROOT / "submission"
# The timing the hidden drives use, as the brief states it.
ACTION_HORIZON_S = 4.0  # how long one answer's command may hold authority
AGE_LIMIT_S = 2.5  # the oldest a command may be when it arrives
MINIMUM_INTERVAL_S = 0.5  # the fastest DriveBench asks; agent.yaml may raise it
REQUEST_TIMEOUT_S = 6.0
CRUISE_SPEED_MPS = 50 / 9  # 20 km/h
# About how long Gemini took to answer in the course's drives. cloud-check
# measures yours; change it to see what a slower answer does to your timing.
ANSWER_LATENCY_S = 1.4
# The request budget of the drives below. Each hidden drive has its own, which
# your policy reads from settings.request_budget.
BUDGET = 60
# Your closed-loop event logs, from `run ... --submission submission --event-log`,
# relative to the repository root. The last section reads them.
EVENT_LOGS = []

RequestPolicy = load_request_policy(SUBMISSION)


def drive_config(budget):
    """The fixture demo's configuration at the hidden drives' timing, after your agent.yaml."""
    base = load_config(PROJECT_ROOT / "configs" / "demo-vla-fixture.yaml")
    base = replace(
        base,
        vla=replace(
            base.vla,
            action_horizon_s=ACTION_HORIZON_S,
            minimum_interval_s=MINIMUM_INTERVAL_S,
            request_timeout_s=REQUEST_TIMEOUT_S,
            fixture=replace(base.vla.fixture, path=str(PROJECT_ROOT / base.vla.fixture.path)),
        ),
        command_validation=replace(base.command_validation, maximum_command_age_s=AGE_LIMIT_S),
        controller=replace(base.controller, target_speed_mps=CRUISE_SPEED_MPS),
    )
    config = apply_submission_overlay(base, SUBMISSION)
    # The budget is the instructors' to set, never agent.yaml's.
    return replace(config, vla=replace(config.vla, maximum_requests_per_run=budget))


settings = RequestPolicySettings.from_vla(drive_config(BUDGET).vla)
print(f"{type(RequestPolicy(settings)).__module__}.RequestPolicy, with {settings}")

## The road

Three lanes, the car in the middle one, 45 s at the cruise speed. A car stands
stopped in its lane 95 m ahead, so it comes into the measured 50 m about 8 s in.
The car slows on the approach, changes into the left lane at 12 s, and passes
it. A truck stands stopped in that lane 205 m from the start; the car slows
again, changes back at 30 s, and passes the truck too. Every change of what the
car measures is an **event**, as the brief defines one: an object coming within
50 m in the car's lane or a lane beside it, or a lane beside the car becoming
clear or occupied.

Edit `OBJECTS`, `LANE_CHANGES`, or `SLOWER` to try another road. The script
drives the car whatever your policy does, so a road only changes what your
policy is shown, not what the car does about it.

In [ ]:
LANE_WIDTH_M = 3.5
DETECTION_RADIUS_M = 50.0
DURATION_S = 45.0
# Stopped objects: (id, kind, distance from the start in m, lane: -1 left, 0 middle, 1 right).
OBJECTS = [("stopped-car", "vehicle", 95.0, 0), ("stopped-truck", "vehicle", 205.0, -1)]
# The lane the car is in from each time on, in s.
LANE_CHANGES = [(0.0, 0), (12.0, -1), (30.0, 0)]
# Where the car slows: (from s, until s, lowest speed in m/s), down and back up again.
SLOWER = [(8.5, 12.0, 3.5), (27.5, 30.0, 3.5)]
RELATIONS = {0: LaneRelation.SAME, -1: LaneRelation.LEFT, 1: LaneRelation.RIGHT}


def speed_at(now_s):
    for start_s, end_s, lowest_mps in SLOWER:
        if start_s <= now_s < end_s:
            middle_s = (start_s + end_s) / 2
            if now_s < middle_s:
                share = (now_s - start_s) / (middle_s - start_s)
                return CRUISE_SPEED_MPS + (lowest_mps - CRUISE_SPEED_MPS) * share
            share = (now_s - middle_s) / (end_s - middle_s)
            return lowest_mps + (CRUISE_SPEED_MPS - lowest_mps) * share
    return CRUISE_SPEED_MPS


def position_at(now_s, step_s=0.01):
    return sum(speed_at(index * step_s) * step_s for index in range(round(now_s / step_s)))


def lane_at(now_s):
    return [lane for start_s, lane in LANE_CHANGES if start_s <= now_s][-1]


def scene_at(now_s):
    """The scene DriveBench measures at now_s: every object within 50 m."""
    travelled_m, lane, speed_mps = position_at(now_s), lane_at(now_s), speed_at(now_s)
    objects = []
    for object_id, kind, distance_m, object_lane in OBJECTS:
        ahead_m, left_m = distance_m - travelled_m, (lane - object_lane) * LANE_WIDTH_M
        if math.hypot(ahead_m, left_m) > DETECTION_RADIUS_M:
            continue
        relation = RELATIONS.get(object_lane - lane, LaneRelation.UNKNOWN)
        in_path = relation is LaneRelation.SAME and ahead_m > 0.0
        objects.append(
            TrackedObject(
                object_id=object_id,
                kind=kind,
                relative_position_m=(ahead_m, left_m),
                relative_velocity_mps=(-speed_mps, 0.0),
                length_m=4.5,
                width_m=1.8,
                lane_relation=relation,
                in_path=in_path,
                path_distance_m=ahead_m if in_path else None,
                path_relative_velocity_mps=-speed_mps if in_path else None,
            )
        )
    return LocalScene(
        timestamp_s=now_s,
        ego_speed_mps=speed_mps,
        ego_length_m=4.5,
        ego_width_m=1.8,
        lane_offset_m=0.0,
        heading_error_rad=0.0,
        objects=tuple(objects),
        left_lane_available=lane > -1,
        right_lane_available=lane < 1,
    )


def lane_clearance(scene):
    """Whether each lane beside the car is clear, as DriveBench judges it for a request."""
    judged = withhold_scene(scene, {"objects"})
    return judged.left_lane_clear, judged.right_lane_clear


def events(drive):
    """[(time, what changed)]: every event of the drive, as the brief defines one."""
    found, previous = [], None
    for tick in drive.ticks:
        scene = scene_at(tick.now_s)
        near = {
            item.object_id
            for item in scene.objects
            if item.lane_relation in (LaneRelation.SAME, LaneRelation.LEFT, LaneRelation.RIGHT)
        }
        clear = lane_clearance(scene)
        if previous is not None:
            for object_id in sorted(near - previous[0]):
                found.append((tick.now_s, f"{object_id} within 50 m"))
            for side, before, now in zip(("left", "right"), previous[1], clear):
                if before != now:
                    state = {True: "clear", False: "occupied", None: "unknown"}[now]
                    found.append((tick.now_s, f"{side} lane {state}"))
        previous = (near, clear)
    return found

## The brief's rules, applied to a drive

The cell below writes out three of the brief's rules, so you can see what they
make of your policy before Gemini is asked anything:

- **Waste.** A request is wasted when it is sent while the last answer's
  command still holds, within its horizon, and nothing has changed since that
  answer's request: the same objects within 50 m, the same lane clearance, the
  car's speed within 0.5 m/s, and no fault beginning or ending. A request sent
  within one age limit, 2.5 s, of that horizon's end is never wasted, since its
  answer could not arrive before the horizon ends. During a fault, a request
  sent less than 5 s after the one before it is wasted.
- **Reaction.** The delay from each event to the next request: 1 at 1.0 s or
  less, falling evenly to 0 at 3.0 s, and averaged over the drive's events.
- **Fault conduct.** The delay from the end of an outage to the next request:
  1 at 3.0 s or less, falling evenly to 0 at 10 s.

Grading applies them to Gemini drives, each run three times, as the brief
says; here they apply to one scripted drive. A figure here is no forecast of
your mark: it shows what your policy does on this road.

In [ ]:
WASTE_SPEED_MPS = 0.5
FAULT_SPACING_S = 5.0
REACTION_FULL_S, REACTION_NONE_S = 1.0, 3.0
RESUMPTION_FULL_S, RESUMPTION_NONE_S = 3.0, 10.0
# The checks: an event is answered promptly when it is asked about as soon as
# the request in flight allows, within one answer and one control step.
REACT_WITHIN_S = ANSWER_LATENCY_S + 0.1
# and a policy resumes in time when it asks within one fault spacing of an
# outage's end, and the control step it is collected on.
RESUME_WITHIN_S = FAULT_SPACING_S + 0.1


def drive(*, budget=BUDGET, faults=(), policy=None):
    """One timing drive of your policy, or of ``policy``, on the road above."""
    config = drive_config(budget)
    if policy is None:
        policy = RequestPolicy(RequestPolicySettings.from_vla(config.vla))
    return drive_request_policy(
        policy,
        config,
        scene_at,
        duration_s=DURATION_S,
        answer_latency_s=ANSWER_LATENCY_S,
        faults=faults,
    )


def falling(delay_s, full_s, none_s):
    """1 at full_s or sooner, 0 at none_s or later, evenly between."""
    if delay_s is None:
        return 0.0
    return max(0.0, min(1.0, (none_s - delay_s) / (none_s - full_s)))


def fault_changed(drive, from_s, to_s):
    """Whether a fault began or ended after from_s and by to_s."""
    edges = [edge for window in drive.faults for edge in (window.start_s, window.end_s) if edge is not None]
    return any(from_s < edge <= to_s for edge in edges)


def wasted(drive):
    """{request_id: why it was wasted} by the brief's rule."""
    reasons = {}
    for request in drive.requests:
        windows = [window for window in drive.faults if window.covers(request.sent_s)]
        if windows:
            earlier = [
                other.sent_s
                for other in drive.requests
                if other.sent_s < request.sent_s and any(window.covers(other.sent_s) for window in windows)
            ]
            if earlier and request.sent_s - max(earlier) < FAULT_SPACING_S:
                reasons[request.request_id] = (
                    f"{request.sent_s - max(earlier):.1f} s after the last request in a fault"
                )
            continue
        answers = [
            other
            for other in drive.requests
            if other.failure is None and other.collected_s is not None and other.collected_s <= request.sent_s
        ]
        if not answers:
            continue
        last = answers[-1]
        horizon_end_s = last.sent_s + drive.action_horizon_s
        if request.sent_s >= horizon_end_s or horizon_end_s - request.sent_s <= AGE_LIMIT_S:
            continue
        before, now = last.scene, request.scene
        if (
            {item.object_id for item in before.objects} == {item.object_id for item in now.objects}
            and lane_clearance(before) == lane_clearance(now)
            and abs(before.ego_speed_mps - now.ego_speed_mps) <= WASTE_SPEED_MPS
            and not fault_changed(drive, last.sent_s, request.sent_s)
        ):
            reasons[request.request_id] = (
                f"{horizon_end_s - request.sent_s:.1f} s left of the command, nothing changed"
            )
    return reasons


def reactions(drive):
    """[(time, what changed, delay to the next request or None)] for every event."""
    sent = sorted(request.sent_s for request in drive.requests)
    found = []
    for event_s, label in events(drive):
        later = [sent_s for sent_s in sent if sent_s >= event_s - 1e-9]
        found.append((event_s, label, round(later[0] - event_s, 3) if later else None))
    return found


def resumptions(drive):
    """[(outage end, delay to the next request or None)] for every outage that ends."""
    sent = sorted(request.sent_s for request in drive.requests)
    found = []
    for window in drive.faults:
        if window.kind is FaultKind.PROVIDER_OUTAGE and window.end_s is not None:
            later = [sent_s for sent_s in sent if sent_s >= window.end_s - 1e-9]
            found.append((window.end_s, round(later[0] - window.end_s, 3) if later else None))
    return found


def gate_checks(drives):
    """The structural gates' own checks of your policy: (gate, passed, detail)."""
    ticks = [tick for drive in drives for tick in drive.ticks]
    problems = [tick for tick in ticks if tick.problem is not None]
    if problems:
        first = problems[0]
        monotone = (False, f"{len(problems)} answers could not stand; the first, at {first.now_s:.1f} s: {first.problem}")
    else:
        monotone = (True, f"{len(ticks)} ticks decided, every answer True or False")
    slowest_s = max(tick.elapsed_s for tick in ticks)
    step_s = drives[0].control_step_s
    if slowest_s <= step_s:
        unblocked = (True, f"every call returned within the {step_s:g} s control step, the slowest in {1000 * slowest_s:.2f} ms")
    else:
        unblocked = (False, f"the slowest call took {1000 * slowest_s:.2f} ms, longer than the {step_s:g} s control step")
    return [(MONOTONE_RESTRICTION, *monotone), (CONTROL_LOOP_UNBLOCKED, *unblocked)]


def report(lines):
    """Print each (name, passed, detail) as PASS or NOT YET, and return whether all passed."""
    for name, passed, detail in lines:
        print(f"{'PASS   ' if passed else 'NOT YET'}  {name}: {detail}")
    return all(passed for _, passed, _ in lines)


def describe(drive, title):
    """Print what the drive's requests were, and what the rules make of them."""
    reasons = wasted(drive)
    refused = [request for request in drive.requests if request.failure is not None]
    print(f"== {title}: {len(drive.requests)} requests, {len(refused)} refused, {len(reasons)} wasted")
    for request in drive.requests:
        if request.failure is not None:
            outcome = f"refused, {request.failure.value}"
        elif request.collected_s is None:
            outcome = "unanswered when the drive ended"
        else:
            outcome = f"answered at {request.collected_s:.1f} s"
        if request.authority is not None:
            outcome += f", its command held {request.authority[0]:.1f} to {request.authority[1]:.1f} s"
        note = f"  WASTED: {reasons[request.request_id]}" if request.request_id in reasons else ""
        print(f"  {request.sent_s:5.1f} s  {outcome}{note}")
    found = reactions(drive)
    for event_s, label, delay_s in found:
        delay = "no request after it" if delay_s is None else f"next request {delay_s:.1f} s later"
        print(f"  event at {event_s:5.1f} s, {label}: {delay}")
    if found:
        mean = sum(falling(delay_s, REACTION_FULL_S, REACTION_NONE_S) for _, _, delay_s in found) / len(found)
        print(f"  reaction, by the brief's rule: {mean:.2f}")
    if drive.requests:
        print(f"  waste share, by the brief's rule: {len(reasons) / len(drive.requests):.2f}")
    for end_s, delay_s in resumptions(drive):
        delay = "never" if delay_s is None else f"{delay_s:.1f} s later"
        print(
            f"  outage ended at {end_s:.1f} s; first request {delay}; fault conduct "
            f"{falling(delay_s, RESUMPTION_FULL_S, RESUMPTION_NONE_S):.2f}"
        )
    plot(timeline(drive, reasons), title)


def timeline(drive, reasons=None):
    """What the plot draws, from a timing drive."""
    return {
        "duration_s": drive.duration_s,
        "faults": [(window.kind.value, window.start_s, window.end_s or drive.duration_s) for window in drive.faults],
        "requests": [
            {
                "sent_s": request.sent_s,
                "collected_s": request.collected_s,
                "refused": request.failure is not None,
                "authority": request.authority,
                "wasted": reasons is not None and request.request_id in reasons,
            }
            for request in drive.requests
        ],
        "events": [event_s for event_s, _ in events(drive)] if reasons is not None else [],
        "remaining": [(tick.now_s, tick.requests_remaining) for tick in drive.ticks],
    }


def plot(line, title):
    """Requests against the commands they bought, fault windows shaded, events dotted."""
    figure, (top, bottom) = plt.subplots(
        2, 1, figsize=(11, 4), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )
    for kind, start_s, end_s in line["faults"]:
        top.axvspan(start_s, end_s, color="0.88")
        top.text(start_s, 2.45, kind.replace("provider_", ""), fontsize=8, va="top")
    for event_s in line["events"]:
        top.axvline(event_s, color="0.4", linestyle=":", linewidth=1)
    for request in line["requests"]:
        end_s = line["duration_s"] if request["collected_s"] is None else request["collected_s"]
        colour = "tab:red" if request["refused"] else "tab:blue"
        top.plot([request["sent_s"], end_s], [2, 2], color=colour, linewidth=2)
        top.plot([request["sent_s"]], [2], marker="x" if request["wasted"] else "o", color=colour)
        if request["authority"] is not None:
            start_s, until_s = request["authority"]
            top.plot([start_s, until_s], [1, 1], color="tab:green", linewidth=6, solid_capstyle="butt")
    top.set_yticks([1, 2], ["command holds", "asked to answered"])
    top.set_ylim(0.5, 2.5)
    top.set_title(f"{title}: x marks a wasted request, red a refused one, dotted lines events")
    times = [now_s for now_s, remaining in line["remaining"] if remaining is not None]
    left = [remaining for _, remaining in line["remaining"] if remaining is not None]
    if left:
        bottom.step(times, left, where="post", color="0.3")
    bottom.set_ylabel("budget left")
    bottom.set_xlabel("time (s)")
    bottom.set_xlim(0, line["duration_s"])
    figure.tight_layout()
    plt.show()

## Your policy on the road

As shipped, `request_policy.py` asks every time DriveBench lets it: as soon as
an answer arrives, it asks again. Each blue line runs from a request to its
answer, and each green bar is the time a command held authority.

In [ ]:
clean = drive()
describe(clean, "the road, no fault")

## Task 1: ask when the scene changes

At `# TODO: keep what the last request saw`,
`# TODO: clear the history that __init__ keeps`, and
`# TODO: ask when the scene changes, not on a timer` in
`submission/request_policy.py`, make the policy notice each change: compare
what `tick.scene` holds with what the last request saw. Lane clearance is
judged from objects: `withhold_scene(tick.scene, {"objects"})` from
`metadrive_starter.vla` gives it as DriveBench judges it for a request, in
`left_lane_clear` and `right_lane_clear`.

The check drives the road twice with one policy object, as a run that drives
two episodes would, so `begin_episode()` must forget the first drive. It
passes when every event is asked about within 1.5 s, as soon as a request in
flight allows. The shipped timer passes it: asking all the time reacts
quickly. Keep it passing through the tasks that follow, which ask less.

In [ ]:
def check_reaction(drive, title):
    late = [(event_s, label, delay_s) for event_s, label, delay_s in reactions(drive) if delay_s is None or delay_s > REACT_WITHIN_S]
    if late:
        event_s, label, delay_s = late[0]
        delay = "never" if delay_s is None else f"{delay_s:.1f} s later"
        return (title, False, f"{len(late)} events asked about late; the first, {label} at {event_s:.1f} s, {delay}")
    return (title, True, f"every event asked about within {REACT_WITHIN_S:.1f} s")


def check(task):
    """Task ``task``'s check, printed as PASS or NOT YET; True when every line passes."""
    if task == 1:
        policy = RequestPolicy(RequestPolicySettings.from_vla(drive_config(BUDGET).vla))
        first, second = drive(policy=policy), drive(policy=policy)
        same = [request.sent_s for request in first.requests] == [request.sent_s for request in second.requests]
        lines = [
            check_reaction(first, "asks when the scene changes"),
            ("begin_episode() clears the history", same, "a second drive asks at the same times" if same else "a second drive asks at other times: something of the first is left over"),
        ]
        drives = [first, second]
    elif task == 2:
        drives = [drive()]
        reasons = wasted(drives[0])
        lines = [
            (
                "holds while the command holds",
                not reasons,
                "no request wasted" if not reasons else f"{len(reasons)} of {len(drives[0].requests)} requests wasted",
            )
        ]
    elif task == 3:
        drives = [drive(faults=[OUTAGE]), drive(faults=[QUOTA])]
        lines = []
        for faulted, title in zip(drives, ("outage", "quota to the end")):
            in_fault = [reason for reason in wasted(faulted).values() if "in a fault" in reason]
            refused = sum(request.failure is not None for request in faulted.requests)
            lines.append(
                (
                    f"backs off under the {title}",
                    not in_fault,
                    f"{refused} requests refused, none sooner than {FAULT_SPACING_S:g} s after the last"
                    if not in_fault
                    else f"{len(in_fault)} of {refused} refused requests sent sooner than {FAULT_SPACING_S:g} s after the last",
                )
            )
        (end_s, delay_s), = resumptions(drives[0])
        lines.append(
            (
                "asks again after the outage",
                delay_s is not None and delay_s <= RESUME_WITHIN_S,
                f"first request {'never' if delay_s is None else f'{delay_s:.1f} s'} after the outage ended at {end_s:g} s",
            )
        )
    elif task == 4:
        drives = [drive(budget=SMALL_BUDGET)]
        last_event_s = events(drives[0])[-1][0]
        left = next(tick.requests_remaining for tick in drives[0].ticks if tick.now_s >= last_event_s - 1e-9)
        lines = [
            (
                "keeps a reserve of the budget",
                left > 0,
                f"{left} of {SMALL_BUDGET} requests left when the last event came, at {last_event_s:.1f} s",
            )
        ]
    else:
        raise ValueError("the tasks are 1 to 4")
    return report(lines + gate_checks(drives))


check(1)

## Task 2: hold while the command holds

At `# TODO: hold while the command that has authority still holds` in
`submission/request_policy.py`, hold the request back while the command in
force still holds and nothing changed: `tick.active` is the answer behind it,
`tick.issued_at_s` when the model was asked for it, and
`settings.action_horizon_s` how long it holds. An answer arrives about 1.4 s
after its request, so a request sent within the last 2.5 s of a command is
never wasted: it is how the next command arrives before this one ends.

The check passes when the road's drive wastes no request by the brief's rule.

In [ ]:
describe(drive(), "the road, no fault")
check(2)

## Task 3: back off through faults

Hidden drives run some scenes a second time with a fault, mid-drive: an
outage, during which every request fails for 10 s, or quota refusals from some
time to the end of the drive. A refused request costs budget as an answered
one does. While no answer arrives, the command in force ends at its horizon
and the backup driving takes over.

At `# TODO: back off after a failure` in `submission/request_policy.py`,
wait after each failure: `tick.failure` says why a request failed, on the tick
its failure arrives, `model_unavailable` during an outage and `quota` for a
refusal. Asking more often than once every 5 s during a fault wastes requests,
and waiting longer finds the provider back late.

The check drives the road with a 10 s outage while the car passes the stopped
car, and with quota refusals from 33 s to the end. It passes when no refused
request comes sooner than 5 s after the one before it, and the policy asks
again within 5.1 s of the outage's end.

In [ ]:
OUTAGE = FaultWindow(FaultKind.PROVIDER_OUTAGE, 14.0, 24.0)
QUOTA = FaultWindow(FaultKind.PROVIDER_QUOTA, 33.0)

describe(drive(faults=[OUTAGE]), "a 10 s outage")
describe(drive(faults=[QUOTA]), "quota refusals to the end")
check(3)

## Task 4: keep a reserve of the budget

Every drive has a request budget, `settings.request_budget`; once it is spent,
the model is never asked again that drive, and the backup driving takes over
when the last command ends. At `# TODO: keep a reserve of the budget` in
`submission/request_policy.py`, keep some of it for what the road has not
shown yet: `tick.requests_remaining` counts what is left, a request in flight
counted as spent.

The check drives the road on a budget of 15 requests. It passes when requests
are left when the road's last event comes.

In [ ]:
SMALL_BUDGET = 15

describe(drive(budget=SMALL_BUDGET), f"a budget of {SMALL_BUDGET}")
check(4)

## Your own drives

A drive in MetaDrive writes every request, answer, failure, and fault to its
event log. List logs in `EVENT_LOGS` in the first cell, and this cell draws
each as above, from what the log holds:

```bash
uv run metadrive-starter run --config configs/demo-vla-fixture.yaml --submission submission --event-log tmp/timing-fixture-01.jsonl
```

A log on [Vertex](../../docs/model-providers.md#vertex-course-provider) shows
your policy with Gemini's own answers and delays; every request there is
billable. A log does not record the scene each request was sent with, so this
cell counts the requests sent while a command still held, which includes every
wasted one but also those sent because something changed. A command's end is
read as its horizon's or the next command's start, whichever came first.

An event log is appended to, so give every run a file of its own; a file that
several runs wrote is drawn run by run. Under each run's first line this cell
prints the pace the run held. A machine that cannot keep up with the pace it
was asked for simulates more slowly, and every answer then arrives younger in
simulated time than it took: such a run flatters your timing, so take no
figure from it.

In [ ]:
from datetime import datetime


def runs_in_event_log(path):
    """An event log's records, one list per run: a log is appended to, so a file
    that several runs wrote holds them all."""
    runs = {}
    for line in Path(path).read_text().splitlines():
        if line.strip():
            record = json.loads(line)
            runs.setdefault(record["run_id"], []).append(record)
    return list(runs.values())


def drive_from_event_log(records):
    """One closed-loop run's requests, answers, commands, and faults, from its records."""
    requests, faults, active, accepted, steps = {}, [], {}, [], []
    duration_s, horizon_s, asked_factor = 0.0, None, None
    for record in records:
        kind, now_s, payload = record["event_type"], record["sim_time_s"], record["payload"]
        if now_s is not None:
            duration_s = max(duration_s, now_s)
        if kind == "run_started":
            horizon_s = payload["config"]["vla"]["action_horizon_s"]
            simulator = payload["config"]["simulator"]
            asked_factor = simulator["realtime_factor"] if simulator["realtime"] else None
        elif kind == "control_applied":
            steps.append((now_s, datetime.fromisoformat(record["wall_time_utc"]).timestamp()))
        elif kind == "inference_submitted":
            requests[payload["request_id"]] = {
                "sent_s": now_s,
                "collected_s": None,
                "refused": False,
                "authority": None,
                "wasted": False,
            }
        elif kind in ("inference_completed", "inference_failed", "inference_discarded"):
            request = requests.get(payload["request_id"])
            if request is not None:
                request["collected_s"] = now_s
                request["refused"] = kind != "inference_completed"
        elif kind == "command_validation":
            command = payload["requested_command"]
            if payload["decision"]["disposition"] != "fallback":
                accepted.append((now_s, command["command_id"], command["issued_at_s"] + command["action_horizon_s"]))
        elif kind == "fault_activated":
            active[payload["fault"]["fault_id"]] = (payload["fault"]["kind"], now_s)
        elif kind == "fault_cleared":
            fault_kind, start_s = active.pop(payload["fault"]["fault_id"])
            faults.append((fault_kind, start_s, now_s))
    faults.extend((fault_kind, start_s, duration_s) for fault_kind, start_s in active.values())
    for (start_s, command_id, end_s), following in zip(accepted, [*accepted[1:], None]):
        if following is not None:
            end_s = min(end_s, following[0])
        if command_id in requests:
            requests[command_id]["authority"] = (start_s, min(end_s, duration_s))
    # Simulated seconds per wall-clock second, over the run's control steps.
    achieved_factor = None
    if len(steps) > 1 and steps[-1][1] > steps[0][1]:
        achieved_factor = (steps[-1][0] - steps[0][0]) / (steps[-1][1] - steps[0][1])
    return {
        "duration_s": duration_s,
        "action_horizon_s": horizon_s,
        "asked_factor": asked_factor,
        "achieved_factor": achieved_factor,
        "faults": faults,
        "requests": sorted(requests.values(), key=lambda request: request["sent_s"]),
        "events": [],
        "remaining": [],
    }


def describe_log(line, title):
    requests = line["requests"]
    refused = sum(request["refused"] for request in requests)
    held = [
        request
        for request in requests
        if any(
            other["authority"] is not None and other["authority"][0] <= request["sent_s"] < other["authority"][1]
            for other in requests
        )
    ]
    print(
        f"== {title}: {len(requests)} requests over {line['duration_s']:.1f} s, {refused} refused, "
        f"{len(held)} sent while a command still held; a command's horizon is {line['action_horizon_s']} s"
    )
    asked, achieved = line["asked_factor"], line["achieved_factor"]
    if asked is None:
        print("  not paced: simulated time ran as fast as the machine allowed, so no delay here is a real one")
    elif achieved is not None:
        print(f"  simulated at {achieved:.2f} of real time, asked for {asked:g}")
        if achieved < 0.95 * asked:
            print(
                "  this machine did not keep up: every answer arrived younger in simulated time "
                "than it took, so take no timing figure from this run"
            )
    for fault_kind, start_s, end_s in line["faults"]:
        inside = sum(start_s <= request["sent_s"] < end_s for request in requests)
        print(f"  {fault_kind} from {start_s:.1f} to {end_s:.1f} s: {inside} requests sent")
    plot(line, title)


for path in EVENT_LOGS:
    runs = runs_in_event_log(PROJECT_ROOT / Path(path))
    for number, records in enumerate(runs, start=1):
        title = Path(path).name if len(runs) == 1 else f"{Path(path).name}, run {number} of {len(runs)}"
        describe_log(drive_from_event_log(records), title)
if not EVENT_LOGS:
    print("No event logs given: list some in EVENT_LOGS, in the first cell, and rerun this cell.")